<a href="https://colab.research.google.com/github/Ranjith0805/AI-Project/blob/main/AI_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
!pip install google-generativeai streamlit pyngrok -q

In [ ]:
import google.generativeai as genai
from google.colab import userdata

# Get API key from secrets
api_key = userdata.get('GEMINI_API_KEY')

# Connect to Gemini
genai.configure(api_key=api_key)

# Choose model
model = genai.GenerativeModel('gemini-2.0-flash')

print("Gemini connected successfully!")

In [28]:
%%writefile app.py

import streamlit as st
import google.generativeai as genai
import os
import json

# Configure Gemini
genai.configure(api_key=os.environ.get('GEMINI_API_KEY'))

# Define model
model = genai.GenerativeModel('gemini-2.0-flash')

# Sidebar
with st.sidebar:
    st.title("📚 About")
    st.write("Welcome to Siri's Study Buddy!")
    st.write("Ask me anything and I'll explain it simply!")
    st.divider()
    subject = st.selectbox(
        "📖 Choose Subject",
        ["General", "Mathematics", "Physics", "Chemistry", "Biology", "History", "Computer Science"]
    )
    difficulty = st.selectbox(
        "🎯 Choose Difficulty",
        ["Easy", "Medium", "Hard"]
    )
    st.divider()
    if st.button("🎯 Generate Quiz"):
        quiz_prompt = f"""Generate 5 {difficulty} level multiple choice questions on {subject}.
Return ONLY a JSON array like this, nothing else:
[
  {{
    "question": "question here",
    "options": ["A. option1", "B. option2", "C. option3", "D. option4"],
    "answer": "A. option1"
  }}
]"""
        response = model.generate_content(quiz_prompt)
        raw = response.text.strip()
        raw = raw.replace("```json", "").replace("```", "").strip()
        try:
            st.session_state.quiz = json.loads(raw)
            st.session_state.quiz_answers = {}
            st.session_state.quiz_submitted = False
        except:
            st.error("Quiz generation failed! Try again.")
        st.rerun()
    st.divider()
    st.subheader("📝 Summarizer")
    text_to_summarize = st.text_area("Paste text here to summarize:")
    if st.button("✨ Summarize!"):
        if text_to_summarize:
            summary_prompt = f"Summarize this text in simple points:\n\n{text_to_summarize}"
            st.session_state.chat_history.append({
                "role": "user",
                "parts": [summary_prompt]
            })
            response = model.generate_content(st.session_state.chat_history)
            ai_reply = response.text
            st.session_state.chat_history.append({
                "role": "model",
                "parts": [ai_reply]
            })
            st.rerun()
    st.divider()
    if st.button("🗑️ Clear Chat"):
        st.session_state.chat_history = [
            {
                "role": "user",
                "parts": [f"You are a friendly AI Study Buddy specializing in {subject}. Explain everything simply with examples. At the end always ask if they understood."]
            },
            {
                "role": "model",
                "parts": ["Got it! I'm your Study Buddy! What would you like to learn today? 😊"]
            }
        ]
        st.rerun()

# App title
st.title("🤖 Siri's Study Buddy")
st.subheader("Ask me anything — I'll explain it simply!")

# Initialize chat history
if "chat_history" not in st.session_state:
    st.session_state.chat_history = [
        {
            "role": "user",
            "parts": [f"You are a friendly AI Study Buddy specializing in {subject}. Explain everything simply with examples. At the end always ask if they understood."]
        },
        {
            "role": "model",
            "parts": ["Got it! I'm your Study Buddy! What would you like to learn today? 😊"]
        }
    ]

# Show quiz if generated
if "quiz" in st.session_state and st.session_state.quiz:
    st.subheader("📝 Quiz Time!")
    for i, q in enumerate(st.session_state.quiz):
        st.write(f"**Q{i+1}. {q['question']}**")
        st.session_state.quiz_answers[i] = st.radio(
            f"Select answer for Q{i+1}:",
            q["options"],
            key=f"q{i}"
        )
    if st.button("✅ Submit Quiz"):
        score = 0
        for i, q in enumerate(st.session_state.quiz):
            if st.session_state.quiz_answers.get(i) == q["answer"]:
                score += 1
        st.success(f"🎉 Your Score: {score}/5!")
        if score == 5:
            st.balloons()
            st.write("Perfect score! Amazing!! 🌟")
        elif score >= 3:
            st.write("Good job! Keep it up! 💪")
        else:
            st.write("Keep practicing! You'll get there! 😊")
        st.session_state.quiz = []

# Show chat messages
st.divider()
st.subheader("💬 Chat")
for message in st.session_state.chat_history[2:]:
    if message["role"] == "user":
        st.chat_message("user").write(message["parts"][0])
    else:
        st.chat_message("assistant").write(message["parts"][0])

# Input box
user_input = st.chat_input("Ask your question here...")

if user_input:
    st.session_state.chat_history.append({
        "role": "user",
        "parts": [user_input]
    })
    response = model.generate_content(st.session_state.chat_history)
    ai_reply = response.text
    st.session_state.chat_history.append({
        "role": "model",
        "parts": [ai_reply]
    })
    st.rerun()

Overwriting app.py


In [30]:
from google.colab import userdata
import os
import subprocess
from pyngrok import ngrok

# Kill any old tunnels
ngrok.kill()

# Set tokens from secrets
os.environ['GEMINI_API_KEY'] = userdata.get('GEMINI_API_KEY')
ngrok.set_auth_token(userdata.get('NGROK_TOKEN'))

# Start Streamlit
process = subprocess.Popen(
    ["streamlit", "run", "app.py", "--server.port", "8501"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    env=os.environ
)

# Create public link
public_url = ngrok.connect(8501)
print(f"Your app is live at: {public_url}")

Your app is live at: NgrokTunnel: "https://casualty-fondness-barber.ngrok-free.dev" -> "http://localhost:8501"
